# Анализ эксперимента по подстройке коэффициентов Kp и Ki от 26 ноября 2025


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nn_laser_stabilizer.config.config import load_config
from nn_laser_stabilizer.utils.paths import get_experiment_dir

EXPERIMENT_NAME = "pid_delta_tuning"
EXPERIMENT_DATE = "2025-11-26"
EXPERIMENT_TIME = "15-53-18"

EXPERIMENT_DIR = get_experiment_dir(
    experiment_name=EXPERIMENT_NAME,
    experiment_date=EXPERIMENT_DATE,
    experiment_time=EXPERIMENT_TIME,
)
CONFIG_PATH = EXPERIMENT_DIR / "config.yaml"
ENV_LOG_PATH = EXPERIMENT_DIR / "env_logs" / "env.log"
TRAIN_LOG_PATH = EXPERIMENT_DIR / "logs" / "train.log"
CONNECTION_LOG_PATH = EXPERIMENT_DIR / "connection_logs" / "connection.log"

config = load_config(CONFIG_PATH)
print(f"Эксперимент: {config.experiment_name}")
print(f"Seed: {config.seed}")

## Анализ логов окружения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_keyval_log

# env.log содержит строки двух типов: шаги (есть should_reset) и reset (нет).
# parse_keyval_log читает все строки; ниже воспроизводим прежнюю логику:
#   - классификация step/reset по наличию should_reset;
#   - reset-строки наследуют номер последнего шага (как current_step).
_raw = parse_keyval_log(ENV_LOG_PATH)
_raw = _raw[
    _raw[["kp", "ki", "kd", "error_mean_norm", "error_std_norm"]].notna().all(axis=1)
].reset_index(drop=True)

is_step = _raw["should_reset"].notna()

env_df = pd.DataFrame({
    "Step": _raw["step"],
    "time": _raw["time"],
    "Type": np.where(is_step, "step", "reset"),
    "Kp": _raw["kp"],
    "Ki": _raw["ki"],
    "Kd": _raw["kd"],
    "Delta Kp": _raw["delta_kp_norm"].where(is_step),
    "Delta Ki": _raw["delta_ki_norm"].where(is_step),
    "Error mean norm": _raw["error_mean_norm"],
    "Error std norm": _raw["error_std_norm"],
    "Reward": _raw["reward"].where(is_step),
    "Should reset": _raw["should_reset"].where(is_step, other=True).astype(bool),
})
env_df["Step"] = env_df["Step"].ffill().fillna(0).astype(int)
reset_steps = env_df.loc[env_df["Type"] == "reset", "Step"].tolist()

print(f"Загружено {len(env_df)} записей из логов окружения")
print(f"Найдено {len(reset_steps)} reset событий")
print(f"Диапазон шагов: {env_df['Step'].min()} - {env_df['Step'].max()}")

In [ ]:
step_df = env_df[env_df['Type'] == 'step'].copy()
print("=== Статистика по шагам окружения ===")
print(step_df.describe())


In [ ]:
exploration_steps = config.training.exploration_steps
initial_collect_steps = config.training.initial_collect_steps
neural_network_step = max(initial_collect_steps, exploration_steps)

columns_to_plot = ['Kp', 'Ki', 'Error mean norm', 'Error std norm', 'Reward']

for col in columns_to_plot:
    plt.figure(figsize=(12, 5))
    plt.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=0.8, label=col)
    
    for reset_step in reset_steps:
        if reset_step <= step_df['Step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= step_df['Step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title(f'{col} over Steps')
    plt.xlabel('Step')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
nn_step_df = step_df[step_df['Step'] >= neural_network_step].copy()

if len(nn_step_df) > 0:
    plt.figure(figsize=(12, 5))
    plt.plot(nn_step_df['Step'], nn_step_df['Error std norm'], alpha=0.8, linewidth=0.8)
    
    for reset_step in reset_steps:
        if reset_step >= neural_network_step and reset_step <= nn_step_df['Step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5, label='Reset' if reset_step == reset_steps[0] else '')
    
    plt.title('Error std norm over Steps (NN phase)')
    plt.xlabel('Step')
    plt.ylabel('Error std norm')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=step_df['Kp'], y=step_df['Error std norm'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error Std Norm')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=step_df['Kp'], y=step_df['Error mean norm'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Kp')
cur_ax.set_ylabel('Error mean norm')
cur_ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cur_ax = axes[0]
sns.scatterplot(x=step_df['Ki'], y=step_df['Error std norm'], alpha=0.6, ax=cur_ax)
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error Std Norm')
cur_ax.grid(True, alpha=0.3)

cur_ax = axes[1]
sns.scatterplot(x=step_df['Ki'], y=step_df['Error mean norm'], alpha=0.6, ax=cur_ax, color='orange')
cur_ax.set_xlabel('Ki')
cur_ax.set_ylabel('Error mean norm')
cur_ax.grid(True, alpha=0.3)


In [ ]:
corr_matrix = step_df[['Kp', 'Ki', 'Error mean norm', 'Error std norm', 'Reward']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


## Анализ процесса обучения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_train_log

# Новый формат: step time loss_q1 loss_q2 [actor_loss] buffer_size.
# Фильтруем строки-метрики и приводим к историческому набору колонок
# (actor_loss = NaN там, где актор не обновлялся).
loss_df = parse_train_log(TRAIN_LOG_PATH)
loss_df = (
    loss_df[loss_df[["loss_q1", "loss_q2", "buffer_size"]].notna().all(axis=1)]
    .reindex(columns=["step", "loss_q1", "loss_q2", "actor_loss", "buffer_size"])
    .reset_index(drop=True)
)
print(f"Загружено {len(loss_df)} записей из логов обучения")
print(f"Диапазон шагов обучения: {loss_df['step'].min()} - {loss_df['step'].max()}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(loss_df['step'], loss_df['loss_q1'], 'b-', alpha=0.7, label='Q1 Loss')
axes[0].set_title('Q1 Loss')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_df['step'], loss_df['loss_q2'], 'g-', alpha=0.7, label='Q2 Loss')
axes[1].set_title('Q2 Loss')
axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(loss_df['step'], loss_df['loss_q1'] + loss_df['loss_q2'], 'r--', alpha=0.7, label='Sum (Q1 + Q2)')
axes[2].set_title('Sum (Q1 + Q2)')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Loss')
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
actor_loss_df = loss_df[loss_df['actor_loss'].notna()]
if len(actor_loss_df) > 0:
    plt.plot(actor_loss_df['step'], actor_loss_df['actor_loss'], 'r-', alpha=0.7)
    plt.title('Actor Loss')
else:
    plt.text(0.5, 0.5, 'No actor loss data', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Actor Loss (no data)')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(loss_df['step'], loss_df['buffer_size'], 'm-', alpha=0.7)
plt.title('Buffer Size')
plt.xlabel('Step')
plt.ylabel('Size')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Анализ состояния установки


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_connection_log

# parse_connection_log отдаёт сырые записи SEND/READ (тип — в колонке 'event').
# Историческое поведение: i-я посылка (SEND) спаривается с i-м чтением (READ).
conn_raw = parse_connection_log(CONNECTION_LOG_PATH)

send_df = conn_raw[conn_raw["event"] == "SEND"][
    ["kp", "ki", "kd", "control_min", "control_max"]
].reset_index(drop=True)
send_df["step"] = range(len(send_df))

read_df = conn_raw[conn_raw["event"] == "READ"][
    ["process_variable", "control_output"]
].reset_index(drop=True)
read_df["step"] = range(len(read_df))

connection_df = send_df.merge(read_df, on="step", how="left")
print(f"Загружено {len(connection_df)} записей из логов соединения")
if len(connection_df) > 0:
    print(f"Диапазон шагов: {connection_df['step'].min()} - {connection_df['step'].max()}")

In [ ]:
neural_network_step = max(initial_collect_steps, exploration_steps) * config.env.args.block_size
if len(connection_df) > 0:
    setpoint = config.env.args.setpoint
    
    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')
    plt.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint ({setpoint})')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('Process Variable')
    plt.xlabel('Step')
    plt.ylim(500, 1700)
    plt.ylabel('Process Variable')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('Control Output')
    plt.xlabel('Step')
    plt.ylabel('Control Output')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['kp'], 'r-', alpha=0.7, linewidth=0.8, label='Kp')
    plt.plot(connection_df['step'], connection_df['ki'], 'g-', alpha=0.7, linewidth=0.8, label='Ki')
    plt.plot(connection_df['step'], connection_df['kd'], 'b-', alpha=0.7, linewidth=0.8, label='Kd')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('PID Coefficients')
    plt.xlabel('Step')
    plt.ylabel('Coefficient Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
if len(connection_df) > 0:
    corr_matrix = connection_df[['kp', 'ki', 'control_output', 'process_variable']].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix (Connection Data)')
    plt.tight_layout()
    plt.show()


In [ ]:
step_df = step_df.sort_values('time')
step_df['time_diff'] = step_df['time'].diff()
step_df['time_diff_ms'] = step_df['time_diff'] * 1000 
if len(step_df) > 0:
    step_df['time_relative'] = step_df['time'] - step_df['time'].iloc[0]
    step_df['time_relative_minutes'] = step_df['time_relative'] / 60

time_df, step_time_df = env_df.copy(), step_df.copy()

print("=== Статистика по времени ===")
print(f"Всего записей: {len(time_df)}")
print(f"Шагов: {len(time_df[time_df['Type'] == 'step'])}")
print(f"Reset событий: {len(time_df[time_df['Type'] == 'reset'])}")

if len(step_time_df) > 0:
    print(f"\n=== Статистика интервалов между шагами ===")
    print(step_time_df['time_diff_ms'].describe())
    print(f"\nМедианный интервал: {step_time_df['time_diff_ms'].median():.2f} мс")
    print(f"Средний интервал: {step_time_df['time_diff_ms'].mean():.2f} мс")
    print(f"Максимальный интервал: {step_time_df['time_diff_ms'].max():.2f} мс")
    print(f"Минимальный интервал: {step_time_df['time_diff_ms'].min():.2f} мс")
    
    plt.figure(figsize=(12, 5))
    plt.plot(step_time_df['Step'], step_time_df['time_diff_ms'], 'b-', alpha=0.7, linewidth=0.8)
    plt.title('Time Intervals Between Steps')
    plt.xlabel('Step')
    plt.ylabel('Time Interval (ms)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 5))
    plt.hist(step_time_df['time_diff_ms'].dropna(), bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Time Intervals Between Steps')
    plt.xlabel('Time Interval (ms)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()